# **Algorithmic Recourse: Практика**

**Привет, друзья.**

Через все уроки модуля «Deep Dive в Counterfactuals» шла Тамара: 63 года, бывшая бухгалтерка, подала заявку на позицию ML-инженера. Скрининговая модель приглашает на собеседование при $p_{\text{invite}} \ge 0.5$, а Тамаре дала $p = 0.03$. Признаки анкеты: возраст 63, пройдено курсов 2, проектов в портфолио 3, коммерческого опыта 0.5 года. Возраст изменить нельзя, и модель на него почти не смотрит; курсы, проекты и опыт Тамара может нарастить.

В уроках каждый метод отвечал Тамаре по отдельности. Здесь мы соберем ответы в одном месте и сравним их по свойствам из шага «Метрики и главный эмпирический факт» урока «Реализации и выводы»: validity, proximity, sparsity, plausibility, actionability, robustness.

В этой практике вы:
- соберете данные и модель-лес из уроков и проверите, что у Тамары $p = 0.03$;
- найдете минимальное изменение по Wachter перебором по сетке значений и увидите, как вес $\lambda$ решает, станет ли контрфактуал валидным;
- получите с DiCE три разных плана с замороженным возрастом и посчитаете близость, разреженность и разнообразие;
- добавите притяжение к прототипу класса «приглашена» и посмотрите, как контрфактуал становится типичнее и дальше от Тамары;
- построите FACE своими руками (k-NN-граф, плотность KDE, Дейкстра) и сравните путь шагами с прыжком Wachter;
- зададите стоимость действий в духе Ustun, найдете самый дешевый план и случай, когда recourse нет;
- проверите каждый найденный план на 15 переобученных моделях и на шуме исполнения, как в ROAR и PRR;
- сведете все в одну таблицу свойств;
- повторите Growing Spheres из урока «Growing Spheres и DiCE» и посмотрите, как сид и шаг радиуса меняют найденную точку;
- обучите маленький условный VAE и поищете контрфактуал в его латенте, как REVISE и C-CHVAE;
- зададите причинную модель «курсы → проекты → опыт» и найдете самый дешевый план causal recourse в духе Karimi;
- напишете генетический поиск в духе GeCo и сравните число вызовов модели с полным перебором;
- соберете таблицу свойств заново, уже со всеми методами модуля.

С DiCE на другой задаче (регрессия цен домов) работает тетрадь `PRACTICE_counterfactual_explanations`; эта тетрадь — про свойства recourse на одном человеке.

**Приятного кодинга!**

### Какие библиотеки мы используем и почему не другие

В уроках модуля названы четыре табличные библиотеки. На Colab (Python 3.13, numpy 2.1.3) из них работает одна, поэтому остальные методы мы пишем коротко сами — по формулам из уроков.

| Библиотека | Что в ней есть | На Colab |
|---|---|---|
| `dice-ml` 0.12 | DiCE: genetic, random, kdtree | ставится, используем |
| `alibi` 0.9.6 | Wachter, `CounterfactualProto` | установка откатывает numpy 2.1.3 → 1.26.4, Pillow → 10.4.0, transformers 5.18.0 → 4.57.6 и blis → 0.7.11 и ломает остальное окружение; Wachter и прототипы пишем сами перебором по сетке значений |
| CARLA (`carla-recourse` 0.0.5) | Growing Spheres, REVISE, C-CHVAE, FACE, ROAR, causal recourse | не ставится: требует tensorflow 1.14, для которой нет сборок под Python 3.13 (и numpy 1.19, torch 1.7); FACE, проверку устойчивости, Growing Spheres, REVISE, C-CHVAE и causal recourse пишем сами (VAE для REVISE и C-CHVAE — на torch) |
| `GeCo.jl` | GeCo: генетический поиск с ограничениями PLAF | пакет на Julia, из Python на Colab не вызывается; генетический поиск в духе GeCo пишем на numpy |
| OmniXAI 1.3.1 | обертка над DiCE и другими объяснениями | установка откатывает numpy до 1.23.5 и scikit-learn до 1.1.3 |
| `actionable-recourse` 1.0.1 | Ustun: целочисленное программирование | ставится вместе с `cplex` и `mip==1.12`, но падает при первом вызове `Flipset`: обращается к `np.float_`, которого нет в numpy 2; к тому же метод рассчитан на линейную модель, а у Тамары лес. Стоимость действий считаем полным перебором целочисленных действий |

Все это проверено установкой в окружение с версиями пакетов Colab на 10.10.2026.

In [ ]:
!pip install dice-ml==0.12 -q

In [ ]:
import random, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
import dice_ml

pd.set_option('display.precision', 3)

## 1. Данные и модель из уроков

Код повторяет шаг «Growing Spheres для Тамары» урока «Growing Spheres и DiCE»: те же 2500 синтетических анкет с сидом 0, тот же скоринг (возраст в нем не участвует), та же модель — стандартизация и случайный лес из 200 деревьев. Мы только завернули генерацию в функцию `make_data(seed)`: в разделе 7 она понадобится для переобучения модели на новых данных, как функция `make(seed)` из шага «Устойчивый recourse для Тамары» урока «Обеспечение устойчивости: Robust Recourse (ROAR / PRR)». При `seed=0` получаются ровно те же данные, что в уроках.

In [ ]:
cont = ["age", "courses", "projects", "exp"]
skill = ["courses", "projects", "exp"]           # что Тамара может менять

def make_data(seed, n=2500):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "age":      rng.normal(42, 12, n).clip(22, 70),
        "courses":  rng.normal(4, 2, n).clip(0, 15),
        "projects": rng.normal(3, 2, n).clip(0, 20),
        "exp":      rng.normal(3, 2.5, n).clip(0, 20)})
    score = 0.5*df.courses + 0.7*df.projects + 0.6*df.exp + rng.normal(0, 1, n)  # возраст НЕ участвует
    df["invited"] = (score > np.median(score)).astype(int)
    return df

def make_model(df):
    return Pipeline([
        ("sc", ColumnTransformer([("s", StandardScaler(), cont)])),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=0))
    ]).fit(df[cont], df["invited"])

df = make_data(0)
model = make_model(df)

tamara = np.array([63., 2., 3., 0.5])              # возраст, курсы, проекты, опыт
def p(x, m=model): return m.predict_proba(pd.DataFrame([x], columns=cont))[0, 1]
def p_many(X, m=model): return m.predict_proba(pd.DataFrame(X, columns=cont))[:, 1]

print("p(Тамара приглашена) =", round(p(tamara), 3))
print("доля приглашенных в данных:", df.invited.mean())
df.head()

Проверим обещание «модель на возраст не смотрит». В скоринге возраста нет, но лес все равно может делать по нему разбиения на шуме.

In [ ]:
imp = model.named_steps["clf"].feature_importances_
print("важность признаков в лесу:", dict(zip(cont, imp.round(3).tolist())))
print("p при другом возрасте и тех же навыках:",
      {a: round(float(p([a, 2., 3., 0.5])), 3) for a in (25, 40, 63, 70)})

Лес отдал возрасту около 9% важности, но $p$ Тамары почти не зависит от возраста: от 0.025 до 0.05 на всем диапазоне. Значит, ответ «помолодейте» Тамаре не поможет, даже если бы был возможен, — и возраст во всех методах ниже заморожен на 63.

### Как мы будем мерить свойства

Каждый найденный контрфактуал $x'$ мы опишем одними и теми же числами (их определения — в шаге «Метрики и главный эмпирический факт» урока «Реализации и выводы»):

- **validity** — $p(x') \ge 0.5$ у текущей модели;
- **proximity** — расстояние $L_1$ по навыкам, каждый навык поделен на свой MAD (медианное абсолютное отклонение), как у Wachter; меньше — ближе к Тамаре;
- **sparsity** — сколько из трех навыков изменилось;
- **plausibility** — среднее расстояние до 5 ближайших реальных анкет в стандартизованных навыках; меньше — типичнее (у самой Тамары это 0.177);
- **actionability** — возраст не тронут и ни один навык не уменьшился; для таких планов считаем стоимость в месяцах усилий (раздел 6);
- **robustness** — доля переобученных моделей и доля зашумленных исполнений, при которых план остается валидным (раздел 7).

In [ ]:
mad = (df[skill] - df[skill].median()).abs().median().values
sd = df[skill].std().values
knn = NearestNeighbors(n_neighbors=5).fit(df[skill].values / sd)
unit_cost = np.array([1., 2., 12.])             # месяцев на +1 курс, +1 проект, +1 год опыта (раздел 6)

def proximity(x):    return np.abs((x[1:] - tamara[1:]) / mad).sum()
def sparsity(x):     return int((np.abs(x[1:] - tamara[1:]) > 1e-9).sum())
def plausibility(x): return knn.kneighbors([x[1:] / sd])[0].mean()
def actionable(x):   return bool(x[0] == tamara[0] and (x[1:] >= tamara[1:] - 1e-9).all())
def cost(x):         return float((x[1:] - tamara[1:]) @ unit_cost)

def describe(x):
    return {"courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
            "p": round(p(x), 3), "valid": p(x) >= 0.5, "proximity": round(proximity(x), 2),
            "sparsity": sparsity(x), "plausibility": round(plausibility(x), 3),
            "actionable": actionable(x)}

found = {}                                       # сюда складываем контрфактуалы всех методов
pd.DataFrame([describe(tamara)], index=["Тамара"])

## 2. Wachter: минимальное изменение

Базовая постановка из шага «Что мы уже знаем о контрфактуалах?» урока «Вводный урок: напоминание Counterfactual explanations»:

$$\arg\min_{x'}\ \max_{\lambda}\ \lambda\big(f(x') - y'\big)^2 + d(x, x'),$$

где $d$ — наша proximity ($L_1$ с масштабом по MAD), $y' = 1$. В Alibi эту задачу решает градиентный спуск (для черного ящика градиент оценивается численно), и тот же поиск лежит в основе `CounterfactualProto`. У леса градиента нет, а навыков всего три, поэтому мы заменим спуск полным перебором по сетке значений с шагом 0.1: курсы от 0 до 8, проекты от 0 до 9, опыт от 0 до 8 лет, возраст 63. Это около 600 тысяч точек, лес считает их за несколько секунд.

In [ ]:
t0 = time.time()
c, pr, e = np.meshgrid(np.arange(0, 8.01, 0.1), np.arange(0, 9.01, 0.1), np.arange(0, 8.01, 0.1), indexing="ij")
G = np.column_stack([np.full(c.size, 63.), c.ravel(), pr.ravel(), e.ravel()])
PG = p_many(G)
DG = (np.abs(G[:, 1:] - tamara[1:]) / mad).sum(1)       # proximity каждой точки сетки
print(len(G), "точек, %.1f с" % (time.time() - t0))

Сначала посмотрим, что делает вес $\lambda$: при фиксированном $\lambda$ минимизируем $\lambda(p(x') - 1)^2 + d(x, x')$ по всей сетке.

In [ ]:
rows = []
for lam in [1, 10, 100, 1000]:
    k = np.argmin(lam * (PG - 1)**2 + DG)
    rows.append({"lambda": lam, **describe(G[k])})
pd.DataFrame(rows).set_index("lambda")

При $\lambda = 1$ штраф за прогноз дешевле любого шага, и оптимум — сама Тамара с $p = 0.03$: контрфактуал невалиден. С ростом $\lambda$ точка уходит за границу, а при больших $\lambda$ — глубже в область приглашения, дальше от Тамары. Поэтому в формуле стоит $\max_\lambda$: вес растят, пока прогноз не станет нужным. В пределе это задача «самая близкая точка среди валидных».

**Задание 1.** Найдите среди точек сетки с $p \ge 0.5$ точку с минимальной proximity `DG` и сохраните ее в `x_wachter`.

*Вопрос для самопроверки:* какие навыки меняет контрфактуал Wachter, какое у него $p$ и сколько навыков он трогает? Сравните с Growing Spheres из шага «Growing Spheres для Тамары» урока «Growing Spheres и DiCE» (курсы $2\to2.2$, проекты $3\to4.4$, опыт $0.5\to1.6$).

<details><summary>Проверить себя</summary>

Контрфактуал Wachter: курсы 2 (без изменений), проекты $3 \to 4.4$, опыт $0.5 \to 1.6$ года, $p = 0.505$, proximity 1.69; меняются 2 навыка из 3. Growing Spheres в уроке «Growing Spheres и DiCE» пришел почти в ту же точку (проекты 4.4, опыт 1.6), но тронул и курсы ($2 \to 2.2$): там точку искали случайным сэмплированием в шаре, а полный перебор с расстоянием $L_1$ оставляет курсы на месте. Решение: `x_wachter = G[np.argmin(np.where(PG >= 0.5, DG, np.inf))]`.
</details>

In [ ]:
x_wachter = ...  # Ваш код здесь
found["Wachter"] = x_wachter
pd.DataFrame([describe(x_wachter)], index=["Wachter"])

Контрфактуал Wachter стоит на самой границе: $p$ едва выше 0.5. Это свойство минимального решения, и в разделе 7 мы увидим, во что оно обходится.

## 3. DiCE: несколько разных планов

Сначала повторим шаг «DiCE для Тамары» урока «Growing Spheres и DiCE» как есть: генетический поиск, три контрфактуала, возраст исключен из `features_to_vary`, запрос записан целыми числами. В уроке поиск шел без сида, и третий контрфактуал менялся от запуска к запуску; здесь мы фиксируем `random` и `numpy`, и все три плана воспроизводятся. Рядом с колонкой `invited`, которую печатает DiCE, поставим $p$ от самой модели.

In [ ]:
d = dice_ml.Data(dataframe=df, continuous_features=cont, outcome_name="invited")
m = dice_ml.Model(model=model, backend="sklearn")
query_int = pd.DataFrame([{"age": 63, "courses": 2, "projects": 3, "exp": 0.5}])   # как в уроке

random.seed(0); np.random.seed(0)
cf_int = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query_int, total_CFs=3, desired_class=1, features_to_vary=skill)
out = cf_int.cf_examples_list[0].final_cfs_df.copy()
print(out.dtypes.to_dict())
out["p модели"] = p_many(out[cont].values.astype(float)).round(3)
out

Первые два плана совпадают с таблицей урока: курсы 3, проекты 3, опыт 3.2 и курсы 2, проекты 4, опыт 1.6. DiCE пишет у них `invited = 1`, а модель дает $p = 0.285$ и $p = 0.255$: оба плана **невалидны**. Причина в типах: в запросе курсы и проекты записаны целыми числами, и DiCE приводит найденные контрфактуалы к типам запроса, отбрасывая дробную часть уже после поиска. Внутри поиска курсов было, например, 3.4, а в выдаче осталось 3, и точка вернулась за границу. Отсюда правило: валидность контрфактуала проверяют прогнозом модели, а не колонкой, которую печатает библиотека.

Запишем запрос дробными числами и повторим поиск.

In [ ]:
query = pd.DataFrame([{"age": 63., "courses": 2., "projects": 3., "exp": 0.5}])

random.seed(0); np.random.seed(0)
t0 = time.time()
cf_gen = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill)
print("genetic: %.2f с" % (time.time() - t0))
dice_gen = cf_gen.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_gen], index=["DiCE-1", "DiCE-2", "DiCE-3"])

Теперь все три плана валидны. Первые два — те же пути, что в уроке, но без отброшенной дробной части: DiCE-1 — курсы 3.4 и опыт 3.2 (в уроке курсов осталось 3), DiCE-2 — курсы 2.5, проекты 4.8 и опыт 1.6 (в уроке курсов 2 и проектов 4). Третий план — 5.5 курса при 2.9 проекта: DiCE уменьшил число проектов. `features_to_vary` заморозил возраст, но направление изменений не ограничивает; такой план формально валиден, а выполнить его нельзя (проект из портфолио не «отменяют»). Для этого в DiCE есть `permitted_range`.

Для сравнения запустим тот же запрос методом `random`: он сэмплирует точки в допустимой области и оставляет валидные.

In [ ]:
random.seed(0); np.random.seed(0)
cf_rnd = dice_ml.Dice(d, m, method="random").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill, random_seed=0)
dice_rnd = cf_rnd.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_rnd], index=["random-1", "random-2", "random-3"])

**Задание 2.** Разнообразие набора DiCE меряет определителем матрицы схожести (шаг «Формальная постановка DiCE» урока «Growing Spheres и DiCE»). Посчитаем более простую величину: среднее попарное расстояние $L_1$ с масштабом по MAD между планами набора. Допишите функцию `diversity(X)` для массива планов `X` (строки — планы, столбцы — `cont`).

*Вопрос для самопроверки:* у какого метода, `genetic` или `random`, выше средняя proximity планов и выше diversity набора? Какие числа получились?

<details><summary>Проверить себя</summary>

`genetic`: средняя proximity 2.55, diversity 3.42; `random`: средняя proximity 8.19, diversity 10.82. Метод `random` дальше от Тамары и разнообразнее: он сэмплирует всю допустимую область и не притягивает планы к объекту. Решение: среднее по парам $i < j$ от `np.abs((X[i, 1:] - X[j, 1:]) / mad).sum()`.
</details>

In [ ]:
def diversity(X):
    # Ваш код здесь
    ...

for name, X in [("genetic", dice_gen), ("random", dice_rnd)]:
    print("%-8s средняя proximity = %.2f   средняя sparsity = %.2f   diversity = %.2f"
          % (name, np.mean([proximity(x) for x in X]), np.mean([sparsity(x) for x in X]), diversity(X)))

Теперь запретим уменьшать навыки: `permitted_range` задает коридор, который начинается с текущего значения Тамары.

In [ ]:
random.seed(0); np.random.seed(0)
cf_pr = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill,
    permitted_range={"courses": [2, 15], "projects": [3, 20], "exp": [0.5, 20]})
dice_pr = cf_pr.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
print("diversity = %.2f" % diversity(dice_pr))
pd.DataFrame([describe(x) for x in dice_pr], index=["DiCE+range-1", "DiCE+range-2", "DiCE+range-3"])

С коридором все три плана выполнимы, но меняют только опыт, и diversity набора упала с 3.42 до 0.47. Ограничение по actionability купили ценой diversity: это первый конфликт свойств в тетради.

In [ ]:
found["DiCE-1"], found["DiCE-2"], found["DiCE-3"] = dice_gen
found["DiCE+range"] = dice_pr[0]

## 4. Прототипы: контрфактуал рядом с типичными приглашенными

Прототип строим как в шаге «Прототип и латент для Тамары» урока «Проблема примеров, которых не существует: Prototypes и AE»: среднее 25 ближайших к Тамаре реальных приглашенных анкет (расстояние в стандартизованных признаках), возраст оставляем 63.

In [ ]:
sd_all = df[cont].std().values
inv = df[df.invited == 1][cont].values
proto = inv[np.argsort(np.linalg.norm((inv - tamara) / sd_all, axis=1))[:25]].mean(0)
proto[0] = 63                                   # возраст неизменяем — держим
pd.DataFrame([describe(proto)], index=["прототип"])

Числа совпадают с уроком (курсы 3.3, проекты 3.8, опыт 2.2), и стоит заметить одно: сам прототип модель **не** приглашает, $p = 0.215$. Ближайшие к Тамаре приглашенные получили приглашение отчасти благодаря шуму в скоринге, и их среднее лежит еще до границы. Значит, прототип годится как направление, а не как ответ: его используют как штраф внутри поиска, а валидность обеспечивает условие $p \ge 0.5$.

Возьмем цель прототипного метода из шага «Что добавляется к цели» урока «Проблема примеров, которых не существует: Prototypes и AE» без автоэнкодера (его роль частично играет сам прототип):

$$L(x') = d(x, x') + \theta\,\Big\lVert \frac{x' - \text{proto}}{\text{sd}} \Big\rVert_2^2 \quad\text{при}\quad p(x') \ge 0.5.$$

**Задание 3.** Для каждого $\theta$ из списка найдите на сетке из раздела 2 валидную точку с минимальной $L$. Квадрат стандартизованного расстояния до прототипа для всех точек сетки уже посчитан в `DP`.

*Вопрос для самопроверки:* как меняются proximity и plausibility контрфактуала при переходе от $\theta = 0$ к $\theta = 10$?

<details><summary>Проверить себя</summary>

При $\theta = 0$ (и при $\theta = 1$) ответ совпадает с Wachter: proximity 1.69, plausibility 0.228. При $\theta = 10$: курсы 3.5, проекты 4.2, опыт 2.5, proximity 3.16 (почти вдвое больше), plausibility 0.125 (меньше, то есть типичнее; у самой Тамары 0.177). Решение: `k = np.argmin(np.where(PG >= 0.5, DG + theta * DP, np.inf))`.
</details>

In [ ]:
DP = (((G[:, 1:] - proto[1:]) / sd)**2).sum(1)
rows, proto_cf = [], {}
for theta in [0, 1, 3, 10]:
    k = ...  # Ваш код здесь: индекс точки сетки
    proto_cf[theta] = G[k]
    rows.append({"theta": theta, **describe(G[k]), "до прототипа": round(np.sqrt(DP[k]), 2)})
pd.DataFrame(rows).set_index("theta")

При малых $\theta$ штраф прототипа слабее, чем шаг по сетке, и ответ совпадает с Wachter. При $\theta = 10$ контрфактуал уходит к прототипу и оказывается в более плотной области анкет, но proximity выросла с 1.69 до 3.16, а plausibility-дистанция упала с 0.228 до 0.125. Это компромисс урока «Проблема примеров, которых не существует: Prototypes и AE»: правдоподобность покупается близостью.

In [ ]:
found["Прототип θ=10"] = proto_cf[10]

## 5. FACE руками: путь по плотным областям

CARLA с FACE на Colab не ставится, поэтому повторяем код шага «Три требования к recourse Тамары» урока «Существование и достижимость: FACE, Actionable и Causal Recourse»: вершины графа — Тамара и 2500 реальных анкет (только навыки), ребра — 15 ближайших соседей в стандартизованных навыках, вес ребра $-\log \hat p\big(\tfrac{x_i + x_j}{2}\big)\cdot \lVert x_i - x_j \rVert$ с плотностью KDE в середине ребра. Цель — анкета, которую модель при возрасте Тамары приглашает с $p \ge 0.6$ и которая сама лежит в плотной половине данных. Путь ищет Дейкстра.

Единственное отличие от урока: плотность в серединах ребер мы считаем одним вызовом KDE для всех ребер сразу, а не в цикле. Числа от этого не меняются, а ячейка работает в несколько раз быстрее.

In [ ]:
t0 = time.time()
X = df[skill].values.astype(float)
proba = model.predict_proba(df[cont].assign(age=63.))[:, 1]   # прогноз для навыков анкеты при возрасте Тамары

pts = np.vstack([tamara[1:], X]); N = len(pts)  # Тамара — узел 0
Z = (pts - X.mean(0)) / X.std(0)                # стандартизуем для расстояний
kde = gaussian_kde(X.T)                         # плотность данных

dist, idx = NearestNeighbors(n_neighbors=16).fit(Z).kneighbors(Z)
rows = np.repeat(np.arange(N), 15); cols = idx[:, 1:].ravel(); dd = dist[:, 1:].ravel()
dens = kde(((pts[rows] + pts[cols]) / 2).T)
w = np.maximum(-np.log(dens + 1e-12) * dd, 1e-6)  # -log(плотность) · расстояние
Gr = csr_matrix((w, (rows, cols)), shape=(N, N)); Gr = Gr.maximum(Gr.T)

dens_all = kde(X.T)
targets = np.where((proba >= 0.6) & (dens_all >= np.quantile(dens_all, 0.5)))[0] + 1
dpath, pred = dijkstra(Gr, indices=0, return_predecessors=True)
best = targets[np.argmin(dpath[targets])]

path = [best]
while path[-1] != 0: path.append(pred[path[-1]])
path = path[::-1]
print("%.1f с" % (time.time() - t0))
for k in path: print(np.round(pts[k], 2))
x_face = np.r_[63., pts[best]]
found["FACE"] = x_face
pd.DataFrame([describe(x_face)], index=["FACE"])

Путь совпадает с уроком: Тамара $[2, 3, 0.5] \to [2, 3.55, 0.88] \to [2.18, 4, 1.51] \to [2.39, 4.37, 1.8]$, у конечной анкеты $p = 0.625$ (лес в `describe` дает то же число, ведь конечная точка — навыки реальной анкеты при возрасте 63). Каждая промежуточная точка — реальный человек из выборки.

Сравним, по каким местам проходят путь FACE и прыжок Wachter. Для каждого отрезка возьмем 50 точек и посмотрим самую низкую плотность KDE на нем; чтобы число читалось, переведем его в долю реальных анкет, у которых плотность еще ниже.

In [ ]:
def min_density_share(points):
    ts = np.linspace(0, 1, 50)[:, None]
    low = min(kde((a + ts * (b - a)).T).min() for a, b in zip(points[:-1], points[1:]))
    return (dens_all < low).mean()

face_pts = pts[path]
jump_pts = np.vstack([tamara[1:], x_wachter[1:]])
for name, P in [("FACE", face_pts), ("Wachter", jump_pts)]:
    print("%-8s длина в навыках = %.2f, до конца по прямой = %.2f, "
          "доля анкет с плотностью ниже самой разреженной точки пути = %.3f"
          % (name, sum(np.linalg.norm(b - a) for a, b in zip(P[:-1], P[1:])),
             np.linalg.norm(P[-1] - P[0]), min_density_share(P)))

In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(X[:, 1], X[:, 2], c=proba, cmap="Blues", s=6, vmin=0, vmax=1)
plt.colorbar(sc, label="p(приглашена) при возрасте 63")
plt.plot(face_pts[:, 1], face_pts[:, 2], "-o", color="#eb6834", lw=2, ms=8, label="путь FACE")
plt.plot(jump_pts[:, 1], jump_pts[:, 2], "--s", color="#1baf7a", lw=2, ms=8, label="прыжок Wachter")
plt.scatter([tamara[2]], [tamara[3]], color="black", s=60, zorder=5, label="Тамара")
plt.xlim(1.5, 6.5); plt.ylim(0, 4)
plt.xlabel("проекты"); plt.ylabel("опыт, лет")
plt.title("Анкеты, путь FACE и прыжок Wachter (проекция на проекты и опыт)")
plt.legend(loc="upper right"); plt.show()

На рисунке точки — реальные анкеты в проекции на проекты и опыт, цвет — вероятность приглашения при возрасте 63. Оранжевая ломаная — путь FACE по реальным анкетам, зеленая пунктирная — прыжок Wachter напрямую к границе. Здесь оба маршрута лежат в населенной области: самая разреженная точка пути FACE плотнее, чем у 66% реальных анкет, прыжка Wachter — чем у 55%. Тамара стоит в густой середине данных, и пропасти между ней и приглашенными нет; путь FACE идет по чуть более плотным местам. За реальную конечную точку и реальные промежуточные шаги FACE платит близостью: до его конечной точки 1.93 по прямой в навыках против 1.78 у Wachter, proximity 2.07 против 1.69. Если бы плотной тропы не было, Дейкстра не нашла бы пути к целевым анкетам — это и есть ответ FACE «recourse нет».

Еще одна деталь конечной точки FACE. Цель выбиралась по условию $p \ge 0.6$, но это $p$ дает лес, который видел эту анкету при обучении. Обучим модель заново на тех же данных без одной этой анкеты и спросим ее снова.

In [ ]:
model_wo = make_model(df.drop(index=best - 1))     # узел best — строка best - 1 в df
print("p модели из уроков = %.3f | p модели без этой анкеты = %.3f"
      % (p(x_face), p(x_face, model_wo)))

Без своей анкеты лес дает конечной точке FACE $p = 0.35$, ниже порога. Случайный лес на обучающих объектах завышает уверенность: каждая анкета попала в бутстреп-выборки большинства деревьев. Поэтому порог уверенности $t_p$ в FACE стоит проверять прогнозами, полученными без этого объекта (out-of-fold), — иначе «реальный приглашенный человек» окажется приглашенным только для модели, которая его запомнила. В разделе 7 это проявится еще раз.

## 6. Стоимость действий в духе Ustun

Расстояние в признаках — геометрия, а стоимость — усилие Тамары (шаг «Actionable Recourse» урока «Существование и достижимость: FACE, Actionable и Causal Recourse»). Зададим грубые цены в месяцах: курс — 1 месяц, проект — 2 месяца, год опыта — 12 месяцев (его надо прожить). Множество допустимых действий $A(x)$ — целые числа: от 0 до 6 новых курсов, от 0 до 6 новых проектов и от 0 до 10 полугодий опыта; уменьшать навыки и трогать возраст нельзя. Это 539 действий, и полный перебор по ним заменяет целочисленное программирование из пакета `actionable-recourse`: на конечном множестве он находит точный минимум и так же точно доказывает, что recourse нет.

**Задание 4.** Для каждого действия из `acts` посчитайте его стоимость в месяцах (`act_cost`) и найдите самое дешевое валидное действие.

*Вопрос для самопроверки:* какой план самый дешевый, сколько он стоит месяцев и какое у него $p$? Сколько месяцев стоил бы план Wachter по тем же ценам?

<details><summary>Проверить себя</summary>

Самый дешевый план — четыре новых курса (курсов $2 \to 6$, проекты и опыт без изменений): 4 месяца, $p = 0.59$. План Wachter по тем же ценам стоит 16 месяцев (1.4 проекта по 2 месяца и 1.1 года опыта по 12 месяцев). Решение: `act_cost = acts @ np.array([1, 2, 6])` (полугодие опыта стоит 6 месяцев), `k = np.argmin(np.where(PA >= 0.5, act_cost, np.inf))`.
</details>

In [ ]:
acts = np.array(list(itertools.product(range(7), range(7), range(11))))   # +курсы, +проекты, +полугодия опыта
XA = np.tile(tamara, (len(acts), 1))
XA[:, 1:] += acts * np.array([1., 1., 0.5])
PA = p_many(XA)
act_cost = ...  # Ваш код здесь: стоимость каждого действия в месяцах
k = ...         # Ваш код здесь: индекс самого дешевого валидного действия
x_ustun = XA[k]
found["Ustun"] = x_ustun
print("самый дешевый план:", acts[k], "→", x_ustun, "| стоимость =", act_cost[k],
      "мес. | p =", round(PA[k], 3))
print("стоимость плана Wachter по тем же ценам: %.1f мес." % cost(x_wachter))

order = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))[:6]
pd.DataFrame({"+courses": acts[order, 0], "+projects": acts[order, 1], "+exp": acts[order, 2] * 0.5,
              "месяцев": act_cost[order], "p": PA[order].round(3)})

Самый дешевый план не похож на ближайший. Wachter двигал проекты и опыт: к ним модель чувствительнее (в скоринге у них веса 0.7 и 0.6 против 0.5 у курсов), и на единицу расстояния они дают больший прирост $p$. По усилию же дешевле всего курсы. Proximity и actionability — разные величины, и выбор между ними делает цена, а не геометрия.

Теперь сузим $A(x)$ до реальных ограничений ближайшего полугодия: Тамара успеет не больше двух курсов и двух проектов, а опыт без работы не растет.

In [ ]:
allowed = (acts[:, 0] <= 2) & (acts[:, 1] <= 2) & (acts[:, 2] == 0)
print("допустимых действий:", allowed.sum(), "| из них валидных:", (PA[allowed] >= 0.5).sum(),
      "| максимальное p =", round(PA[allowed].max(), 3))

Ни одно из 9 допустимых действий не дает $p \ge 0.5$: при таких ограничениях recourse для Тамары не существует, и перебор это доказывает, а не предполагает. Так выглядит ответ, ради которого Ustun и соавторы и формулировали задачу через множество действий: «за полгода ничего из доступного не поможет, нужен опыт».

## 7. Устойчивость в духе ROAR и PRR

Recourse — обещание во времени (урок «Обеспечение устойчивости: Robust Recourse (ROAR / PRR)»). Проверим каждый план с двух сторон:

- **сдвиг модели (ROAR):** компания переобучает скрининг на новых данных. Как функция `make(seed)` из шага «Устойчивый recourse для Тамары», генерируем данные с сидами 1–15 и обучаем 15 моделей; доля моделей, у которых $p \ge 0.5$, — robustness к сдвигу;
- **шум исполнения (PRR):** Тамара выполнит план неточно. Добавляем к навыкам гауссов шум со стандартными отклонениями 0.5 курса, 0.5 проекта и 0.7 года опыта (как в уроке), 500 раз; доля валидных исполнений — robustness к шуму.

Рядом поставим запас (margin) $p - 0.5$ и proximity.

In [ ]:
t0 = time.time()
models = [make_model(make_data(s)) for s in range(1, 16)]
print("15 моделей за %.0f с" % (time.time() - t0))

**Задание 5.** Допишите `share_models(x)` — долю из 15 переобученных моделей `models`, у которых $p(x) \ge 0.5$. Функция шума `share_noise(x)` уже написана: она каждый раз берет один и тот же сид, поэтому все планы проверяются на одинаковых возмущениях.

*Вопрос для самопроверки:* какая доля переобученных моделей приглашает Тамару по плану Wachter и по рецепту «с буфером» из урока «Обеспечение устойчивости: Robust Recourse (ROAR / PRR)» (курсы 3, проекты 4, опыт 6.5)? Чему равны их margin?

<details><summary>Проверить себя</summary>

План Wachter приглашают 0 из 15 переобученных моделей (доля 0.0), его margin 0.005. План «с буфером» приглашают 15 из 15 (доля 1.0), его margin 0.475. Решение: `return np.mean([p(x, mm) >= 0.5 for mm in models])`.
</details>

In [ ]:
def share_models(x):
    # Ваш код здесь
    ...

def share_noise(x, n=500):
    g = np.random.default_rng(3)                # шум исполнения по навыкам
    V = np.array(x, float) + g.normal(0, [0, 0.5, 0.5, 0.7], size=(n, 4))
    V[:, 1:] = np.clip(V[:, 1:], 0, None)
    return (p_many(V) >= 0.5).mean()

found["урок ROAR: у границы"] = np.array([63., 3., 4., 3.5])
found["урок ROAR: с буфером"] = np.array([63., 3., 4., 6.5])

true_thr = np.median(0.5*df.courses + 0.7*df.projects + 0.6*df.exp)   # порог скоринга без шума
def score_margin(x): return 0.5*x[1] + 0.7*x[2] + 0.6*x[3] - true_thr

rob = pd.DataFrame({name: {"p": round(p(x), 3), "margin": round(p(x) - 0.5, 3), "proximity": round(proximity(x), 2),
                           "запас по скорингу": round(score_margin(x), 2),
                           "доля моделей": round(share_models(x), 3), "доля при шуме": round(share_noise(x), 3)}
                    for name, x in found.items()}).T
rob

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, col, title in [(ax[0], "доля моделей", "сдвиг модели (15 переобученных)"),
                      (ax[1], "доля при шуме", "шум исполнения (500 исполнений)")]:
    a.scatter(rob["margin"], rob[col], s=60, color="#2a78d6")
    for name, r in rob.iterrows():
        a.annotate(name, (r["margin"], r[col]), fontsize=8, xytext=(4, 3), textcoords="offset points")
    a.set_xlabel("margin = p − 0.5 у текущей модели"); a.set_ylabel("доля валидных")
    a.set_title(title); a.set_ylim(-0.05, 1.08); a.set_xlim(-0.03, 0.62)
plt.tight_layout(); plt.show()

На двух графиках каждая точка — план одного метода: по горизонтали его запас над порогом у текущей модели, по вертикали доля переобученных моделей (слева) и зашумленных исполнений (справа), при которых план остается валидным. План Wachter с margin 0.005 не выживает ни на одной из 15 переобученных моделей и только в 19% зашумленных исполнений; план «с буфером» с margin 0.475 выживает везде, но его proximity 4.99 против 1.69 у Wachter.

Связь margin с устойчивостью при этом грубая: у FACE margin 0.125, а переобученные модели не приглашают его ни разу (конечная точка взята из обучающей выборки, раздел 5), у прототипа при $\theta = 10$ margin 0, а доля моделей 0.533. Данные у нас синтетические, и мы знаем правило, по которому их размечали, поэтому в таблице есть столбец «запас по скорингу» — $0.5\cdot\text{курсы} + 0.7\cdot\text{проекты} + 0.6\cdot\text{опыт}$ минус медиана этой суммы по данным. Доли переобученных моделей идут почти строго по нему: планы с отрицательным запасом по скорингу держатся на шумовом «кармане» текущего леса и при переобучении пропадают. В реальной задаче этого правила нет, поэтому ROAR и PRR проверяют сами возмущения, а не margin одной модели.

Последний шаг — встроить требование устойчивости в сам поиск. На целочисленных действиях из раздела 6 найдем самый дешевый план, валидный **на всех** 15 переобученных моделях и на текущей (ROAR: худшая модель из множества), и самый дешевый план, который выживает хотя бы в 95% зашумленных исполнений (PRR с $\delta = 0.05$).

In [ ]:
t0 = time.time()
worst = np.min([p_many(XA, mm) for mm in models + [model]], axis=0)     # худшая модель для каждого действия
k_roar = np.argmin(np.where(worst >= 0.5, act_cost, np.inf))
cheap = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))
k_prr = next(k for k in cheap if share_noise(XA[k]) >= 0.95)
found["ROAR (худшая из 16)"] = XA[k_roar]
found["PRR (δ=0.05)"] = XA[k_prr]
for name, k in [("Ustun", np.argmin(np.where(PA >= 0.5, act_cost, np.inf))), ("ROAR", k_roar), ("PRR", k_prr)]:
    print("%-6s +%s → %s | стоимость = %d мес. | p = %.3f" % (name, acts[k], XA[k], act_cost[k], PA[k]))
print("%.1f с" % (time.time() - t0))

## 8. Итоговая таблица свойств

Соберем все найденные планы в одну таблицу. Стрелка в заголовке говорит, куда лучше: ↑ — больше, ↓ — меньше. Стоимость в месяцах посчитана только для выполнимых планов (возраст не тронут, ни один навык не уменьшился).

In [ ]:
summary = pd.DataFrame({name: {
    "courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
    "validity ↑": p(x) >= 0.5,
    "proximity ↓": round(proximity(x), 2),
    "sparsity ↓": sparsity(x),
    "plausibility ↓": round(plausibility(x), 3),
    "actionable ↑": actionable(x),
    "месяцев ↓": round(cost(x), 1) if actionable(x) else np.nan,
    "доля моделей ↑": round(share_models(x), 3),
    "доля при шуме ↑": round(share_noise(x), 3)} for name, x in found.items()}).T
summary

Таблица подтверждает главный вывод шага «Метрики и главный эмпирический факт» урока «Реализации и выводы»: ни один метод не лучший по всем столбцам сразу.
- Самая малая proximity у Wachter (1.69), но его не приглашает ни одна из 15 переобученных моделей, а при шуме исполнения он валиден в 18.6% случаев.
- Самая высокая plausibility (дистанция 0.125 и 0.129) у прототипа при $\theta = 10$ и у FACE; FACE при этом не приглашает ни одна переобученная модель, а прототипный план стоит 27.9 месяца.
- Меньше всего навыков меняют DiCE с коридором и Ustun (по одному), но первый стоит 40.8 месяца, а второй, самый дешевый (4 месяца), держится только на 1 из 15 переобученных моделей.
- DiCE-3 валиден, но невыполним: в нем уменьшается число проектов.
- Полную устойчивость к сдвигу модели дают план «с буфером» (75 месяцев) и ROAR (8 месяцев); у ROAR при этом доля валидных исполнений при шуме 0.668, а PRR (тоже 8 месяцев), наоборот, валиден в 99% исполнений и на 11 из 15 моделей.

Выбор для Тамары зависит от того, что ей дороже: быстрее всего — четыре курса по плану Ustun, с риском, что после переобучения скрининга их не хватит; надежнее всего — план ROAR, два курса и три проекта за 8 месяцев, который выдерживает все 16 моделей.

## 9. Growing Spheres: поиск в растущих сферах

В разделе 2 ближайшую валидную точку нашел полный перебор сетки. В уроке «Growing Spheres и DiCE» ту же задачу решал Growing Spheres (Laugel и соавторы, 2017), и здесь мы повторим его код из шага «Growing Spheres для Тамары». Методу нужны только прогнозы модели. Он сэмплирует точки в шаре вокруг Тамары и растит радиус $r$ с шагом 0.05, пока в шар не попадет хотя бы одна точка, которую модель приглашает. Потом метод плотно сэмплирует шар радиуса $r + 0.1$, берет ближайшую к Тамаре по евклидову расстоянию приглашенную точку и разреживает ее: по очереди возвращает навыки к значениям Тамары, если прогноз от этого не меняется. CARLA, где метод реализован, на Colab не ставится (таблица в начале тетради), поэтому код свой, на numpy.

Код урока завернут в функцию `growing_spheres(seed)`, и к нему добавлен счетчик вызовов модели `calls`: в разделе 12 по этому счетчику сравним методы поиска.

**Задание 6.** Допишите цикл роста радиуса: увеличивайте `r` на `step`, пока среди `m` точек `shell(r, m)` не появится хотя бы одна с прогнозом 1, и остановитесь на этом радиусе.

*Вопрос для самопроверки:* на каком радиусе метод остановился, какой контрфактуал нашел и сколько раз обратился к модели? Чем он отличается от контрфактуала Wachter из раздела 2?

<details><summary>Проверить себя</summary>

Радиус $r = 1.95$; контрфактуал: курсы 2.19, проекты 4.37, опыт 1.64, $p = 0.505$, та же точка, что в уроке «Growing Spheres и DiCE» (курсы 2.2, проекты 4.4, опыт 1.6). Вызовов модели 216003: 39 шаров по 4000 точек, 60000 точек в найденном слое и 3 проверки при разреживании. До Тамары 1.79 по евклидову расстоянию в навыках против 1.78 у Wachter, но proximity ($L_1$ с масштабом по MAD) 1.83 против 1.69, и меняются все три навыка, а не два. Решение: в цикле `while r < 20:` строки `cand = shell(r, m); cand = cand[pred(cand) == 1]`, затем `if len(cand): break` и `r += step`.
</details>

In [ ]:
calls = {"n": 0}                                   # счетчик вызовов модели
def pred(X):
    calls["n"] += len(X)
    return (p_many(np.asarray(X, float)) > 0.5).astype(int)   # как model.predict: класс 1 при p > 0.5

vary = [1, 2, 3]                                   # меняем навыки; возраст (0) заморожен

def growing_spheres(seed, step=0.05, m=4000, m_band=60000):
    gen = np.random.default_rng(seed)
    def shell(radius, m):                          # m точек в шаре радиуса radius вокруг Тамары
        dirs = gen.normal(size=(m, len(vary)))
        dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
        rad = gen.uniform(0, radius**len(vary), size=m) ** (1/len(vary))
        pts = np.tile(tamara, (m, 1))
        pts[:, vary] = np.clip(tamara[vary] + dirs*rad[:, None], 0, None)
        return pts
    r = step                                       # растим радиус, пока в шаре не появится точка класса 1
    while r < 20:
        ...  # Ваш код здесь: шар shell(r, m), проверка прогноза pred, выход из цикла или рост r на step
    band = shell(r + 0.1, m_band); band = band[pred(band) == 1]        # плотно сэмплируем найденный слой
    e = band[np.argmin(np.linalg.norm(band[:, vary] - tamara[vary], axis=1))].copy()   # ближайший «враг»
    for j in [vary[k] for k in np.argsort(np.abs(e[vary] - tamara[vary]))]:   # разреживаем: снимаем лишние правки
        t = e.copy(); t[j] = tamara[j]
        if pred([t])[0] == 1: e = t
    return e, r

calls["n"] = 0
x_gs, r_gs = growing_spheres(2)
print("радиус r = %.2f, вызовов модели: %d" % (r_gs, calls["n"]))
print("евклидово расстояние в навыках: Growing Spheres %.2f, Wachter %.2f"
      % (np.linalg.norm(x_gs[1:] - tamara[1:]), np.linalg.norm(x_wachter[1:] - tamara[1:])))
found["Growing Spheres"] = x_gs
pd.DataFrame([describe(x_gs)], index=["Growing Spheres"])

Разреживание не сняло ни одной правки, и причина в норме. Growing Spheres ищет ближайшую точку в евклидовом шаре, а у шара ближайшая к Тамаре точка границы обычно лежит не на оси, и в ней понемногу меняются все навыки. Вернуть курсы к 2, не сдвигая остальное, значит отойти от границы назад, и прогноз снова становится «не приглашена». Wachter в разделе 2 искал по $L_1$, а у этой нормы ближайшие точки чаще лежат на осях, поэтому курсы там остались на месте.

Метод случайный, поэтому проверим, как ответ зависит от сида генератора и от шага радиуса.

In [ ]:
rows = []
for seed in range(10):
    calls["n"] = 0
    x, r = growing_spheres(seed)
    rows.append({"seed": seed, "r": round(r, 2), **describe(x), "вызовов": calls["n"]})
gs_seeds = pd.DataFrame(rows).set_index("seed")
display(gs_seeds)

rows = []
for step in [0.02, 0.05, 0.2, 0.5]:
    calls["n"] = 0
    x, r = growing_spheres(2, step=step)
    rows.append({"шаг радиуса": step, "r": round(r, 2), **describe(x), "вызовов": calls["n"]})
pd.DataFrame(rows).set_index("шаг радиуса")

На 10 сидах радиус остановки от 1.85 до 1.95, проекты от 4.37 до 4.41, опыт от 1.55 до 1.64, а курсы от 2.10 до 2.42: направление ответа устойчиво, а курсы зависят от того, какие точки попали в выборку. Все 10 планов валидны, все трогают три навыка, $p$ от 0.505 до 0.565, proximity от 1.75 до 1.99.

Шаг радиуса задает обмен точности на число вызовов. При шаге 0.02 метод сделал 440003 вызова и нашел точку с курсами 2.00, проектами 4.44 и опытом 1.60, proximity 1.72, почти ответ Wachter; при шаге 0.5 вызовов 76003, а proximity 1.90. Сам метод по построению не дает ни правдоподобности, ни выполнимости: здесь навыки только растут потому, что у Тамары ни один навык не мешает приглашению, а не потому, что метод это проверял.

## 10. REVISE и C-CHVAE: поиск в латенте VAE

Урок «Проблема примеров, которых не существует: Prototypes и AE», шаг «Продолжая идею автоэнкодеров», предлагает искать контрфактуал не в пространстве признаков, а в латенте генеративной модели: декодер порождает объекты, похожие на обучающие данные, поэтому найденная точка лежит на многообразии данных. REVISE (Joshi и соавторы, 2019) спускается по коду $z$ градиентом:

$$z^{*} = \arg\min_{z}\ \mathrm{yloss}\big(f(g(z)), y'\big) + \lambda\, d\big(g(z), x\big),\qquad c = g(z^{*}).$$

C-CHVAE (Pawelczyk и соавторы, 2019) берет условный VAE, кодирует только изменяемые признаки при условии защищенных и ищет ближайший к коду объекта $z_0$ код, чей декодированный объект меняет прогноз, только по прогнозам модели:

$$z^{*}=\arg\min_{z}\ \lVert z-z_0\rVert\quad\text{при}\quad f\big(g(z, x_{\mathrm{prot}})\big)=y'.$$

У Тамары защищенный признак один, возраст. Обучим маленький условный VAE: вход — три стандартизованных навыка, условие — стандартизованный возраст, латент двумерный, 300 эпох Adam. Обучение идет на CPU за несколько секунд; числа этого раздела зависят от обучения и на другом аппаратном обеспечении могут немного отличаться.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
mu_all, sd_all = df[cont].mean().values, df[cont].std().values
Xs = torch.tensor((df[cont].values - mu_all) / sd_all, dtype=torch.float32)   # стандартизованные анкеты
S, A = Xs[:, 1:], Xs[:, :1]                        # навыки (изменяемые) и возраст (условие)
y_t = torch.tensor(df.invited.values, dtype=torch.float32)

class CVAE(nn.Module):                             # условный VAE: кодирует навыки при условии возраста
    def __init__(self, d=3, c=1, k=2, h=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d + c, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 2 * k))
        self.dec = nn.Sequential(nn.Linear(k + c, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, d))
    def encode(self, s, c): return self.enc(torch.cat([s, c], 1)).chunk(2, 1)    # среднее и log-дисперсия
    def decode(self, z, c): return self.dec(torch.cat([z, c], 1))

vae = CVAE()
opt = torch.optim.Adam(vae.parameters(), 1e-3)
t0 = time.time()
for epoch in range(300):
    for b in torch.randperm(len(S)).split(128):
        mu, lv = vae.encode(S[b], A[b])
        z = mu + torch.randn_like(mu) * (0.5 * lv).exp()
        rec = ((vae.decode(z, A[b]) - S[b])**2).sum(1).mean()
        kl = (-0.5 * (1 + lv - mu**2 - lv.exp()).sum(1)).mean()
        loss = rec + 0.5 * kl
        opt.zero_grad(); loss.backward(); opt.step()
print("VAE за %.0f с; ошибка восстановления %.2f, KL %.2f" % (time.time() - t0, rec.item(), kl.item()))

def to_std(x): return torch.tensor(((np.asarray(x, float) - mu_all) / sd_all)[None], dtype=torch.float32)
def from_std(s, age=63.): return np.r_[age, s.detach().numpy()[0] * sd_all[1:] + mu_all[1:]]

xt = to_std(tamara)
with torch.no_grad():
    z0, _ = vae.encode(xt[:, 1:], xt[:, :1])
    x_rec = from_std(vae.decode(z0, xt[:, :1]))
pd.DataFrame([describe(tamara), describe(x_rec)], index=["Тамара", "Тамара после VAE"])

Первое, что стоит проверить у латентного метода, — как VAE восстанавливает саму Тамару. Декодер вернул ей курсы 2.48 и опыт 1.5 вместо 2 и 0.5 года, и модель дает такой анкете $p = 0.255$ вместо 0.03. Причина в данных: три навыка сгенерированы независимо, низкоразмерного многообразия у них нет, и двумерный латент вынужден сжимать три независимых признака. Поиск начинается от кода $z_0$, и любой контрфактуал, найденный в латенте, наследует эту ошибку восстановления. В этом и состоит условие «зависит от качества VAE» из карточки метода.

REVISE нужен градиент классификатора через декодер, а у леса градиента нет. Поэтому для REVISE обучим дифференцируемый суррогат, маленькую MLP на тех же анкетах, а валидность найденных точек проверим лесом.

In [ ]:
torch.manual_seed(0)                               # REVISE нужен дифференцируемый классификатор: учим MLP на тех же анкетах
mlp = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
opt = torch.optim.Adam(mlp.parameters(), 1e-2)
for epoch in range(300):
    loss = nn.functional.binary_cross_entropy_with_logits(mlp(Xs)[:, 0], y_t)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    agree = ((mlp(Xs)[:, 0] > 0).numpy() == (model.predict(df[cont]) == 1)).mean()
    print("MLP: p(Тамара) = %.3f, совпадает с лесом на %.1f%% анкет" % (torch.sigmoid(mlp(xt)).item(), 100 * agree))

mad_std = torch.tensor(mad / sd_all[1:], dtype=torch.float32)    # MAD в стандартизованных единицах

def revise(lam, steps=500, lr=0.05):
    z = z0.clone().requires_grad_(True)
    opt = torch.optim.Adam([z], lr)
    for t in range(steps):
        s = vae.decode(z, xt[:, :1])               # декодированные навыки при возрасте Тамары
        logit = mlp(torch.cat([xt[:, :1], s], 1))[0, 0]
        loss = nn.functional.softplus(-logit) + lam * ((s - xt[:, 1:]).abs() / mad_std).sum()   # yloss + λ·d
        opt.zero_grad(); loss.backward(); opt.step()
    return from_std(vae.decode(z, xt[:, :1])), z.detach().numpy()[0]

revise_x, revise_z = {}, {}
for lam in [0.01, 0.1, 0.3, 1.0]:
    revise_x[lam], revise_z[lam] = revise(lam)
found["REVISE (λ=0.3)"] = revise_x[0.3]
pd.DataFrame([{"lambda": lam, **describe(x), "p у MLP": round(torch.sigmoid(mlp(to_std(x))).item(), 3)}
              for lam, x in revise_x.items()]).set_index("lambda")

MLP совпадает с лесом на 88% анкет. При всех четырех $\lambda$ лес приглашает найденную точку ($p$ от 0.51 до 0.905), но ни одна точка не выполнима: курсы падают с 2 до 0.06–1.29. Целевая функция REVISE не знает, что курсы нельзя «разучить», а декодер вместе с градиентом MLP уводит точку туда, где суррогат уверен. Правдоподобность тоже не получилась: дистанция до 5 ближайших анкет от 0.258 до 0.463 против 0.177 у самой Тамары. При $\lambda = 1$ видно расхождение суррогата и леса: лес дает 0.51, а MLP 0.472, то есть по своему классификатору REVISE до границы не дошел. В итоговую таблицу возьмем $\lambda = 0.3$, где точка валидна для обеих моделей.

C-CHVAE градиент не нужен: он растит кольцо вокруг $z_0$ в латенте, декодирует коды при возрасте 63 и спрашивает у леса прогноз, то есть делает Growing Spheres в латенте.

**Задание 7.** В функции `cchvae` среди валидных кодов кольца выберите ближайший к $z_0$ и верните его анкету `Xc[k]`, код `Z[k]` и радиус `r`.

*Вопрос для самопроверки:* какой план нашел C-CHVAE и какая у него plausibility по сравнению с Growing Spheres и REVISE?

<details><summary>Проверить себя</summary>

В решенной копии C-CHVAE остановился на радиусе 0.60 в латенте и нашел курсы 2.41, проекты 4.34, опыт 1.84 при $p = 0.500$ (ровно 100 из 200 деревьев; условие валидности у нас $p \ge 0.5$). Proximity 2.08 против 1.83 у Growing Spheres, а дистанция до 5 ближайших анкет 0.124 — самая малая в тетради (у Growing Spheres 0.192, у REVISE с $\lambda = 0.3$ 0.258, у FACE 0.129). Числа зависят от обучения VAE и на другом аппаратном обеспечении могут немного отличаться. Решение: `k = np.argmin(np.where(ok, np.linalg.norm(Z - z0.numpy(), axis=1), np.inf))`, затем `return Xc[k], Z[k], r`.
</details>

In [ ]:
def decode_many(Z):                                # латентные коды -> анкеты при возрасте 63
    with torch.no_grad():
        Sd = vae.decode(torch.tensor(Z, dtype=torch.float32), xt[:, :1].repeat(len(Z), 1)).numpy()
    return np.column_stack([np.full(len(Z), 63.), Sd * sd_all[1:] + mu_all[1:]])

def cchvae(seed=0, step=0.05, m=2000):
    gen = np.random.default_rng(seed)
    r = step
    while r < 5:                                   # растим кольцо r - step < |z - z0| <= r в латенте
        d = gen.normal(size=(m, 2)); d /= np.linalg.norm(d, axis=1, keepdims=True)
        Z = z0.numpy() + d * gen.uniform(r - step, r, size=m)[:, None]
        Xc = decode_many(Z)
        ok = p_many(Xc) >= 0.5                     # черный ящик: только прогнозы леса
        if ok.any():
            ...  # Ваш код здесь: индекс k ближайшего к z0 валидного кода и return Xc[k], Z[k], r
        r += step

x_cchvae, z_cchvae, r_cchvae = cchvae()
print("радиус в латенте r = %.2f" % r_cchvae)
found["C-CHVAE"] = x_cchvae
pd.DataFrame([describe(x) for x in [x_gs, revise_x[0.3], x_cchvae]], index=["Growing Spheres", "REVISE (λ=0.3)", "C-CHVAE"])

Посмотрим на оба метода в самом латенте: раскрасим двумерную решетку кодов прогнозом леса для декодированной анкеты и поставим на нее коды реальных анкет.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-3, 3, 121), np.linspace(-3, 3, 121))
Zg = np.column_stack([g1.ravel(), g2.ravel()])
Pg = p_many(decode_many(Zg)).reshape(g1.shape)     # прогноз леса для каждой декодированной точки латента
with torch.no_grad():
    mu_data, _ = vae.encode(S, A)                  # коды реальных анкет

plt.figure(figsize=(7, 5.5))
cs = plt.contourf(g1, g2, Pg, levels=np.linspace(0, 1, 11), cmap="Blues")
plt.colorbar(cs, label="p(приглашена) при возрасте 63")
plt.contour(g1, g2, Pg, levels=[0.5], colors="black", linewidths=1)
plt.scatter(mu_data[:, 0], mu_data[:, 1], s=2, color="gray", alpha=0.3, label="коды реальных анкет")
plt.scatter(*z0.numpy()[0], color="black", s=70, zorder=5, label="код Тамары $z_0$")
plt.scatter(*revise_z[0.3], color="#eb6834", marker="s", s=70, zorder=5, label="REVISE (λ=0.3)")
plt.scatter(*z_cchvae, color="#1baf7a", marker="^", s=90, zorder=5, label="C-CHVAE")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$"); plt.xlim(-3, 3); plt.ylim(-3, 3)
plt.title("Латент VAE: прогноз леса для декодированных анкет")
plt.legend(loc="lower left", fontsize=8); plt.show()

На рисунке цвет — вероятность приглашения у леса для анкеты, которую декодер порождает из данного кода при возрасте 63, черная линия — граница $p = 0.5$, серые точки — коды реальных анкет. Код Тамары лежит в светлой области внутри облака; C-CHVAE сделал короткий шаг через ближайшую границу и остался в облаке, отсюда малая дистанция до реальных анкет. REVISE ушел по градиенту MLP далеко за край облака, в область кодов, из которых декодер порождает нетипичные анкеты. Декодер обещает правдоподобность только там, где он видел данные; за краем облака кода это обещание не работает.

## 11. Causal recourse: действия в причинной модели

В уроке «Существование и достижимость: FACE, Actionable и Causal Recourse» (шаг «Causal Recourse (Karimi)») recourse ищется как вмешательство в структурную причинную модель (SCM):

$$a^{*} = \arg\min_{a}\ \mathrm{cost}(a; x)\quad \text{при}\quad f\big(x^{SCF}_a\big) = y'.$$

Отличие от Ustun в разделе 6 — в $x^{SCF}_a$ вместо $x + a$: итоговое состояние не назначается, а досчитывается по SCM. В шаге «Три требования к recourse Тамары» сказано, что у Тамары проекты и опыт связаны. Зададим SCM вручную:

- курсы $:= U_c$;
- проекты $:= U_p + 0.5\cdot$курсы (каждый второй курс заканчивается проектом в портфолио);
- опыт $:= U_e + 0.25\cdot$проекты (проект дает три месяца опыта).

Генератор данных из раздела 1 этих связей не содержит: SCM описывает, как действия Тамары отзываются в мире, а не как размечали анкеты. Именно это и делает метод трудным на практике: SCM надо задать самим, данных для ее проверки обычно нет. CARLA, где метод реализован, на Colab не ставится, поэтому три шага Перла пишем сами:

1. **абдукция** — восстановить шум $U$ из анкеты Тамары;
2. **действие** — жесткое вмешательство $\mathrm{do}(X_i := x_i + \delta_i)$ заменяет уравнение признака $X_i$, если $\delta_i > 0$;
3. **прогноз** — пересчитать признаки без вмешательства по их уравнениям сверху вниз.

Действия и цены те же, что в разделе 6: 539 целочисленных действий, курс — месяц, проект — два месяца, полугодие опыта — шесть месяцев. Цена считается за само действие, а сдвиги, которые SCM досчитала по причинным связям, бесплатны.

**Задание 8.** Допишите в `scf` проекты и опыт: при вмешательстве — значение Тамары плюс прибавка, без вмешательства — уравнение SCM от уже посчитанного родителя и шума `U_tamara`.

*Вопрос для самопроверки:* какой план causal recourse самый дешевый, сколько он стоит и чем отличается от плана Ustun?

<details><summary>Проверить себя</summary>

Самый дешевый план — три новых курса за 3 месяца: по SCM они дают курсы 5, проекты 4.5 и опыт 0.875 года, $p = 0.72$. План Ustun из раздела 6 — четыре курса за 4 месяца; по SCM он дает $p = 0.92$, то есть переплачивает месяц, потому что не знает, что курсы приносят проекты. Без SCM три курса дали бы $p = 0.395$. Решение: `pr = tamara[2] + delta[1] if do[1] else U_tamara[1] + b_cp * c` и `e = tamara[3] + delta[2] if do[2] else U_tamara[2] + b_pe * pr`.
</details>

In [ ]:
b_cp, b_pe = 0.5, 0.25                             # курсы -> проекты, проекты -> опыт (лет за проект)

def abduct(x):                                     # шаг 1, абдукция: шум U из наблюдаемой анкеты
    c, pr, e = x[1:]
    return np.array([c, pr - b_cp * c, e - b_pe * pr])

U_tamara = abduct(tamara)

def scf(delta, do):                                # шаги 2 и 3: действие do и прогноз потомков
    # delta — прибавки к курсам, проектам и опыту; do — по каким признакам есть вмешательство
    c = tamara[1] + delta[0] if do[0] else U_tamara[0]
    pr = ...  # Ваш код здесь: проекты
    e = ...   # Ваш код здесь: опыт
    return np.array([63., c, pr, e])

D = acts * np.array([1., 1., 0.5])                 # те же 539 целочисленных действий из раздела 6
XS = np.array([scf(d, d > 0) for d in D])          # состояние после действия по SCM
PS = p_many(XS)
k_c = np.argmin(np.where(PS >= 0.5, act_cost, np.inf))
k_u = np.argmin(np.where(PA >= 0.5, act_cost, np.inf))
x_causal = XS[k_c]
found["Causal (SCM)"] = x_causal
print("Causal: действие +%s, стоимость %d мес., состояние после действия %s, p = %.3f"
      % (acts[k_c], act_cost[k_c], np.round(x_causal, 3), PS[k_c]))
print("Ustun:  действие +%s, стоимость %d мес.; по SCM то же действие дает %s, p = %.3f"
      % (acts[k_u], act_cost[k_u], np.round(XS[k_u], 3), PS[k_u]))

order = np.argsort(np.where(PS >= 0.5, act_cost, np.inf))[:6]
pd.DataFrame({"+courses": acts[order, 0], "+projects": acts[order, 1], "+exp": acts[order, 2] * 0.5,
              "месяцев": act_cost[order], "p по SCM": PS[order].round(3), "p без SCM": PA[order].round(3),
              "проекты после": XS[order, 2].round(2), "опыт после": XS[order, 3].round(3)})

В таблице шесть самых дешевых планов по SCM. Четвертая строка необычна: план «+3 курса и +1 проект» стоит 5 месяцев и дает проектов 4.0, меньше, чем «+3 курса» без проекта (4.5). Жесткое вмешательство $\mathrm{do}(\text{проекты} := 3 + 1)$ заменяет уравнение проектов целиком и обрывает стрелку «курсы → проекты», поэтому проекты, которые принесли бы новые курсы, в итог не попадают. Это свойство постановки урока, а не ошибка кода: вмешательство, которое прибавляет к уравнению, а не заменяет его, такого эффекта не дает. Сравним несколько планов в двух мирах — без SCM, как у Ustun, и по SCM.

In [ ]:
def plan_row(name, dc, dp, de):
    d = np.array([dc, dp, de], float)
    x_naive = tamara + np.r_[0, d]
    x_scm = scf(d, d > 0)
    return {"план": name,
            "проекты: Ustun / SCM": f"{x_naive[2]:.2f} / {x_scm[2]:.2f}",
            "опыт: Ustun / SCM": f"{x_naive[3]:.3f} / {x_scm[3]:.3f}",
            "p: Ustun / SCM": f"{p(x_naive):.3f} / {p(x_scm):.3f}"}

display(pd.DataFrame([plan_row("+3 курса", 3, 0, 0), plan_row("+3 курса и +1 проект", 3, 1, 0),
                      plan_row("+2 проекта", 0, 2, 0), plan_row("+2 проекта и +1.5 года опыта", 0, 2, 1.5)]))

allowed = (acts[:, 0] <= 2) & (acts[:, 1] <= 2) & (acts[:, 2] == 0)
print("ограничения полугодия по SCM: допустимых действий %d, валидных %d, максимальное p = %.3f"
      % (allowed.sum(), (PS[allowed] >= 0.5).sum(), PS[allowed].max()))

Два проекта по SCM дают Тамаре еще полгода опыта, но $p$ поднимается только с 0.05 до 0.115. В плане «+2 проекта и +1.5 года опыта» вмешательство в опыт обрывает стрелку «проекты → опыт», и состояние совпадает с наивным: опыт 2.0 года, $p = 0.58$. С ограничениями ближайшего полугодия из раздела 6 recourse нет и по SCM: из 9 допустимых действий валидных 0, максимальное $p = 0.285$ против 0.205 без SCM. Причинные связи сделали два курса полезнее, но не настолько, чтобы хватило.

## 12. GeCo: генетический поиск за интерактивное время

Урок «Ускорение поиска для Real-time: GeCo и CLUE», шаг «GeCo — быстрый поиск без градиентов» (Schleich и соавторы, 2021), обещает контрфактуал за интерактивное время: генетический поиск стартует с кандидатов, меняющих один признак, держит ограничения выполнимости на языке PLAF, отбирает лучших, скрещивает и мутирует их и останавливается, когда лучший кандидат перестает меняться. Пакет `GeCo.jl` написан на Julia, поэтому поиск в духе GeCo пишем на numpy.

На целочисленных действиях раздела 6 полный перебор стоит 539 вызовов модели, и ускорять там нечего. Поэтому возьмем сетку действий в 10 раз мельче: шаг 0.1 курса, 0.1 проекта и 0.1 года опыта, до +12 курсов, +12 проектов и +10 лет опыта, всего $121 \cdot 121 \cdot 101 = 1478741$ действий. Дробный курс — это часть курса; мелкая сетка здесь нужна как модель большого пространства действий, какое бывает, когда признаков и их значений много. Ограничения PLAF у нас те же, что в разделе 6: возраст не меняется, навыки не убывают.

**Задание 9.** Допишите приспособленность `fit`: у валидных кандидатов ($p \ge 0.5$) это стоимость в месяцах, у невалидных — $1000 + 100\cdot(0.5 - p)$, чтобы любой валидный кандидат был лучше любого невалидного, а среди невалидных выше ценились близкие к границе.

*Вопрос для самопроверки:* какой план нашел GeCo с сидом 0, за сколько поколений и вызовов модели?

<details><summary>Проверить себя</summary>

GeCo с сидом 0: +3.2 курса (32 шага по 0.1), 3.2 месяца, $p = 0.510$, 19 поколений и 418 вызовов модели. Время в секундах зависит от аппаратного обеспечения, поэтому сравнивать методы удобнее по числу вызовов. Решение: `fit = np.where(pr >= 0.5, months, 1000 + (0.5 - pr) * 100)`.
</details>

In [ ]:
step_fine = np.array([0.1, 0.1, 0.1])              # сетка в 10 раз мельче: +0.1 курса, +0.1 проекта, +0.1 года
hi_fine = np.array([120, 120, 100])                # до +12 курсов, +12 проектов, +10 лет опыта
price_fine = step_fine * unit_cost                 # месяцев за один шаг по каждому навыку

def p_actions(a):                                  # прогноз для целочисленных шагов a, считает вызовы
    calls["n"] += len(a)
    X = np.tile(tamara, (len(a), 1)); X[:, 1:] += a * step_fine
    return p_many(X)

def geco(seed, pop=40, keep=10, patience=10, max_gen=100):
    g = np.random.default_rng(seed)
    P = np.zeros((pop, 3), int)                    # старт GeCo: кандидаты, меняющие один признак
    j = g.integers(0, 3, pop); P[np.arange(pop), j] = g.integers(1, hi_fine[j] + 1)
    best, gen = [], 0
    while True:
        P = np.unique(P, axis=0)
        pr = p_actions(P)
        months = (P * price_fine).sum(1)
        fit = ...  # Ваш код здесь: месяцы у валидных, 1000 + (0.5 - p) * 100 у невалидных
        order = np.argsort(fit); P, fit = P[order][:keep], fit[order][:keep]   # отбор лучших
        best.append(fit[0]); gen += 1
        if gen >= max_gen or (len(best) > patience and best[-1] == best[-1 - patience]): break
        a, b = P[g.integers(0, len(P), pop)], P[g.integers(0, len(P), pop)]
        child = np.where(g.random((pop, 3)) < 0.5, a, b)                        # кроссовер
        mut = g.random((pop, 3)) < 0.3
        child = np.clip(child + mut * g.integers(-2, 3, (pop, 3)), 0, hi_fine)  # мутация; PLAF: навыки не убывают
        P = np.vstack([P, child])
    return P[0], fit[0], gen

calls["n"] = 0; t0 = time.time()
a_geco, m_geco, gen_geco = geco(0)
t_geco = time.time() - t0
x_geco = tamara + np.r_[0, a_geco * step_fine]
found["GeCo (сетка 0.1)"] = x_geco
print("GeCo: +%s шагов -> %s, %.1f мес., p = %.3f; поколений %d, вызовов модели %d, %.2f с"
      % (a_geco, np.round(x_geco, 2), m_geco, p(x_geco), gen_geco, calls["n"], t_geco))

Сравним с полным перебором той же сетки (полтора миллиона прогнозов леса, блоками по числу курсов) и проверим, как ответ GeCo зависит от правила остановки: `patience` — сколько поколений подряд лучший кандидат может не меняться.

In [ ]:
calls["n"] = 0; t0 = time.time()                   # полный перебор той же мелкой сетки, блоками по курсам
best_m, best_a = np.inf, None
for c0 in range(hi_fine[0] + 1):
    blk = np.array(list(itertools.product([c0], range(hi_fine[1] + 1), range(hi_fine[2] + 1))))
    pr = p_actions(blk); months = (blk * price_fine).sum(1)
    k = np.argmin(np.where(pr >= 0.5, months, np.inf))
    if pr[k] >= 0.5 and months[k] < best_m: best_m, best_a = months[k], blk[k]
t_full = time.time() - t0
print("полный перебор: +%s шагов, %.1f мес.; вызовов модели %d, %.1f с" % (best_a, best_m, calls["n"], t_full))

rows = []
for patience in [3, 10]:
    for seed in range(20):
        calls["n"] = 0
        a, m_, gen = geco(seed, patience=patience)
        rows.append({"patience": patience, "seed": seed, "месяцев": round(m_, 1), "вызовов": calls["n"],
                     "нашел минимум": bool(abs(m_ - best_m) < 1e-9)})
geco_runs = pd.DataFrame(rows)
geco_runs.groupby("patience").agg(доля_минимума=("нашел минимум", "mean"), медиана_месяцев=("месяцев", "median"),
                                  максимум_месяцев=("месяцев", "max"), медиана_вызовов=("вызовов", "median"))

Полный перебор нашел тот же план, +3.2 курса за 3.2 месяца, за 1478741 вызов модели; GeCo с сидом 0 обошелся 418 вызовами, примерно в 3500 раз меньше. Время в секундах зависит от аппаратного обеспечения, но растет вместе с числом вызовов: у полного перебора это секунды, у GeCo доли секунды, а для модели, у которой один прогноз стоит дороже прогноза леса, разница растет в той же пропорции.

Правило остановки решает, насколько GeCo можно верить. При `patience = 10` все 20 сидов нашли минимум 3.2 месяца, медиана 517 вызовов. При `patience = 3` поиск останавливается раньше (медиана 186 вызовов), минимум нашли 7 сидов из 20, медиана стоимости 3.85 месяца, худший сид — 6.4 месяца. Генетический поиск не доказывает, что его план самый дешевый, и не может доказать, что recourse нет: это умеет только полный перебор по множеству действий, как в разделе 6, или точный солвер, как у Ustun.

CLUE из того же урока ищет объект, в котором уверена модель с оценкой неопределенности (байесовская нейросеть), и для точечного леса Тамары не применим, поэтому остается в теории урока. Неопределенность CLUE меряет энтропией прогноза; энтропию прогноза Тамары считает ячейка ниже: она близка к нулю, модель уверенно отказывает, и сомнения, которое CLUE мог бы объяснить, нет.

In [ ]:
# CLUE объясняет неопределенность: энтропия прогноза в битах, H(p) = -p·log2(p) - (1-p)·log2(1-p)
def entropy_bits(q): return -(q * np.log2(q) + (1 - q) * np.log2(1 - q))
print("энтропия прогноза Тамары, бит:", round(entropy_bits(p(tamara)), 3))
print("энтропия пограничной анкеты с p = 0.5, бит:", round(entropy_bits(0.5), 3))

## 13. Итоговая таблица со всеми методами

Соберем таблицу раздела 8 заново, добавив планы разделов 9–12. Для плана causal recourse в столбце месяцев стоит стоимость его действия (три курса), а не сдвиг состояния: проекты и опыт, которые досчитала SCM, Тамара получает без отдельного усилия.

In [ ]:
effort = {name: cost(x) for name, x in found.items()}
effort["Causal (SCM)"] = float(act_cost[k_c])      # усилие causal-плана — его действие, а не сдвиг состояния
summary_all = pd.DataFrame({name: {
    "courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
    "validity ↑": p(x) >= 0.5,
    "proximity ↓": round(proximity(x), 2),
    "sparsity ↓": sparsity(x),
    "plausibility ↓": round(plausibility(x), 3),
    "actionable ↑": actionable(x),
    "месяцев ↓": round(effort[name], 1) if actionable(x) else np.nan,
    "доля моделей ↑": round(share_models(x), 3),
    "доля при шуме ↑": round(share_noise(x), 3)} for name, x in found.items()}).T
summary_all

Новые строки подтверждают вывод раздела 8 и добавляют к нему четыре наблюдения.
- Growing Spheres и GeCo, как Wachter и Ustun, стоят у самой границы и не выживают ни на одной из 15 переобученных моделей; GeCo при этом дает самый дешевый выполнимый план без причинной модели, 3.2 месяца.
- C-CHVAE дает самую малую дистанцию до реальных анкет (0.124), но его $p = 0.500$ лежит на границе, и переобученные модели приглашают его в 1 случае из 15.
- План causal recourse стоит 3 месяца и держится на 11 из 15 переобученных моделей: по SCM три курса приносят еще полтора проекта и 0.375 года опыта. Этот результат верен ровно настолько, насколько верна SCM, которую мы задали сами.
- REVISE с $\lambda = 0.3$ держится на 11 из 15 моделей, но невыполним: он уменьшает курсы.

## **Выводы**

В этой практике мы ответили Тамаре всеми методами модуля на одних данных и одной модели и увидели, что:
- минимальный по расстоянию контрфактуал (Wachter) стоит на границе решения и хрупок при переобучении модели и неточном исполнении;
- валидность надо проверять прогнозом модели: DiCE привел планы к целым типам запроса и напечатал `invited = 1` у невалидных точек;
- DiCE дает меню планов, но `features_to_vary` не запрещает уменьшать навыки; коридор `permitted_range` делает планы выполнимыми и одновременно однообразными;
- прототип задает направление к типичным приглашенным, но сам может оказаться невалидным; притяжение к нему повышает правдоподобность ценой близости;
- FACE ведет по реальным анкетам и показывает путь шагами, а платит за это близостью; порог уверенности для его цели стоит проверять прогнозами модели, обученной без этой анкеты;
- стоимость действий меняет ответ сильнее, чем метрика расстояния, и полный перебор по множеству действий умеет доказать, что recourse нет;
- запас над порогом покупает устойчивость к сдвигу модели и шуму исполнения, и платит за нее близостью и месяцами;
- Growing Spheres приходит почти в ту же точку, что Wachter, но ищет по евклидову расстоянию и поэтому трогает все три навыка; точность и число вызовов модели задает шаг радиуса;
- поиск в латенте VAE (C-CHVAE) дает самый типичный план тетради, но качество ответа ограничено самим VAE: двумерный латент восстанавливает Тамару с ошибкой, а REVISE с градиентом суррогатной MLP уходит из облака реальных анкет и уменьшает курсы;
- причинная модель меняет и стоимость, и сам план: если курсы приносят проекты, а проекты опыт, Тамаре хватает трех курсов вместо четырех, а жесткое вмешательство в проекты обрывает вклад курсов;
- генетический поиск в духе GeCo находит тот же самый дешевый план за сотни вызовов модели вместо полутора миллионов, но минимальности не доказывает и зависит от правила остановки;
- ни один метод не лучший по всем свойствам сразу: выбор зависит от того, ошибиться в каком свойстве дороже в вашей задаче.